In [70]:
# Change your team name to Group_xx
TEAM_NAME = 'G_02'

In [71]:
# NOTE: you CAN change this cell
# If you want to use your own database, download it here
# !gdown ...

In [72]:
# NOTE: you CAN change this cell
# Add more to your needs
# you must place ALL pip install here
!pip install editdistance
!pip install daachorse

In [73]:
# NOTE: you CAN change this cell
# import your library here
import re
import time
import unicodedata
import daachorse
import editdistance
from collections import defaultdict

In [74]:
# NOTE: you MUST change this cell
# New methods / functions must be written under class Solution.
class Solution:
    # True  -> exact text
    # False -> ""
    RAW_FALLBACK = False

    # OCR stuff
    _CONF = str.maketrans({
        "0": "o",
        "1": "i",
        "5": "s",
        "8": "b"
    })

    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str]
    ):
        self.editdistance = editdistance

        self.district_re = re.compile(
            r"^(?:quận|huyện|thị\s*xã|thành\s*phố\s*thủ\s*đức|tx|q|h)(?:\.\s*|\s+|(?=\d)|$)",
            re.I,
        )

        self.prov_prefix = ["thành phố", "tỉnh", "tp", "t"]
        self.ward_prefix = ["thị trấn", "phường", "xã", "tt", "p", "x"]

        self.prov_fuzzy_1 = ("tinh", "tp")
        self.prov_fuzzy_2 = ("thanhpho",)
        self.ward_fuzzy_1 = ("phuong", "xa", "tt")
        self.ward_fuzzy_2 = ("thitran",)
        self.street_fuzzy_1 = ("duong", "pho", "ngo", "ngach", "hem", "kiet", "lo")
        self.street_fuzzy_2 = ("dailo",)

        self.kw_split_re = re.compile(
            r"\s+(?=(?:phường|xã|thị\s+trấn|quận|huyện|tỉnh|thành\s+phố)\b)",
            re.I
        )

        self.km_re = re.compile(r"^km\s*\d+(?:[+.,]\d+)?\s*", re.I)
        self.kiot_re = re.compile(r"^kiot\s+\S+\s*", re.I)
        self.alley_re = re.compile(
            r"^(?:ngõ|ngách|hẻm|kiệt|hom)\s+(?:số\s+)?(?:\S*\d\S*|[a-z])\s+",
            re.I
        )
        self.house_re = re.compile(
            r"^(?:số\s+)?\d+[A-Za-z]?(?:\s*[/\-]\s*\d+[A-Za-z]?)*\s+(?=\S)",
            re.I
        )
        self.stray_re = re.compile(r"^[A-Za-z]\s+(?=(?:đường|phố|đ\.))", re.I)
        self.tail_re = re.compile(r"\s+(?:kéo\s*dài|nối\s*dài|mở\s*rộng)$", re.I)
        self.prefix_re = re.compile(r"^(?:(?:đường|phố|đại\s*lộ)\b\s*|đ\.\s*)", re.I)
        self.strong_re = re.compile(
            r"^(?:(?:đường|phố|đại\s*lộ|quốc\s*lộ|tỉnh\s*lộ|đèo|dốc)\b|đ\.)",
            re.I
        )
        self.hamlet_re = re.compile(
            r"^(?:thôn|bản|ấp|buôn|tổ|khu|lán|trạm|kiot|căn|tầng|nhà|chung\s*cư|cột\s*mốc|lô|rừng|tiểu)\b",
            re.I
        )
        self.note_re = re.compile(r"^(?:ghi\s*chú|lưu\s*ý|note)\b", re.I)

        self.prov_tab = self._build(list_province)
        self.ward_tab = self._build(list_ward)
        self.street_tab = self._build([s for s in list_street if s and s.strip()])

    # ---------------------------------------------------------
    # NORMALIZATION

    def _fold(self, s: str) -> str:
        s = unicodedata.normalize("NFD", str(s).casefold())
        s = "".join(c for c in s if unicodedata.category(c) != "Mn").replace("đ", "d")
        s = "".join(c if c.isalnum() else " " for c in s)
        return " ".join(s.split())

    def _compact(self, s: str) -> str:
        return self._fold(s).replace(" ", "")

    def _pres(self, s: str) -> str:
        return unicodedata.normalize("NFC", str(s)).lower().replace(" ", "")

    @staticmethod
    def _bigrams(s: str) -> set[str]:
        if len(s) < 2:
            return {s} if s else set()
        return {s[i:i + 2] for i in range(len(s) - 1)}

    # ---------------------------------------------------------
    # DATABASE INDEX (EXACT + BIGRAM INVERTED INDEX)

    def _build(self, items):
        exact = {}
        lst = []
        bg_index = defaultdict(list)

        for it in items:
            if not it:
                continue
            it = str(it).strip()
            if not it:
                continue
            c = self._compact(it)
            if not c:
                continue

            p = self._pres(it)
            tc = c.translate(self._CONF)
            exact.setdefault(c, []).append((it, p))

            tokens = self._fold(it).split()
            last = tokens[-1] if tokens else ""
            idx = len(lst)
            lst.append((
                it,
                c,
                p,
                tc,
                last if last.isdigit() else None
            ))

            if len(items) > 300:
                for bg in self._bigrams(tc):
                    bg_index[bg].append(idx)

        return {
            "exact": exact,
            "list": lst,
            "bg_index": bg_index if len(items) > 300 else None
        }

    # ---------------------------------------------------------
    # EXACT MATCHING

    def _exact(self, text: str, tab) -> str:
        c = self._compact(text)
        if not c:
            return ""
        hits = tab["exact"].get(c)
        if not hits:
            return ""
        p = self._pres(text)
        for orig, pre in hits:
            if pre == p:
                return orig
        return hits[0][0]

    # ---------------------------------------------------------
    # OPTIMAL STRING ALIGNMENT DISTANCE

    @staticmethod
    def _osa(a: str, b: str) -> int:
        la, lb = len(a), len(b)
        d = [[0] * (lb + 1) for _ in range(la + 1)]
        for i in range(la + 1):
            d[i][0] = i
        for j in range(lb + 1):
            d[0][j] = j
        for i in range(1, la + 1):
            for j in range(1, lb + 1):
                cost = 0 if a[i - 1] == b[j - 1] else 1
                v = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
                if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                    v = min(v, d[i - 2][j - 2] + 1)
                d[i][j] = v
        return d[la][lb]

    # ---------------------------------------------------------
    # EDIT DISTANCE THRESHOLD

    @staticmethod
    def _md(tl: int) -> int:
        if tl <= 3:
            return 0
        if tl <= 6:
            return 1
        if tl <= 11:
            return 2
        if tl <= 18:
            return 3
        return 4

    # ---------------------------------------------------------
    # FUZZY MATCHING

    def _fuzzy(self, text: str, tab, maxd_fn) -> str:
        q0 = self._compact(text)
        if len(q0) < 3:
            return ""

        p = self._pres(text)
        qs = [q0]
        if "rn" in q0:
            qs.append(q0.replace("rn", "m"))

        tokens = self._fold(text).split()
        q_tail = tokens[-1] if (tokens and tokens[-1].isdigit()) else None

        lst = tab["list"]
        bg_index = tab["bg_index"]

        if bg_index is not None:
            counts = defaultdict(int)
            for q in qs:
                qc = q.translate(self._CONF)
                for bg in self._bigrams(qc):
                    for idx in bg_index.get(bg, ()):
                        counts[idx] += 1
            if not counts:
                return ""
            if len(counts) > 150:
                cand_indices = sorted(counts, key=counts.get, reverse=True)[:150]
            else:
                cand_indices = counts.keys()
            candidates = (lst[i] for i in cand_indices)
        else:
            candidates = lst

        best = None

        for orig, t, tp, tc, t_tail in candidates:
            tl = len(t)
            md = maxd_fn(tl)
            if md == 0:
                continue

            if q_tail and t_tail and q_tail != t_tail:
                continue

            best_d = None
            for q in qs:
                qc = q.translate(self._CONF)
                if abs(len(qc) - len(tc)) > md:
                    continue

                d = self.editdistance.eval(qc, tc)

                if d == md + 1:
                    d = min(d, self._osa(qc, tc))

                if d <= md and (d / tl) <= 0.36:
                    if best_d is None or d < best_d:
                        best_d = d

            if best_d is None:
                continue

            key = (
                best_d / tl,
                best_d,
                abs(len(q0) - tl),
                self.editdistance.eval(p, tp)
            )
            if best is None or key < best[0]:
                best = (key, orig)

        return best[1] if best else ""

    # ---------------------------------------------------------
    # ADDRESS SPLITTING

    def _split(self, text: str) -> list[str]:
        primary = re.split(r"[,;|\r\n\t~]+|::+|-->+", text)
        parts = [" ".join(p.split()) for p in primary]
        parts = [p for p in parts if p]
        if len(parts) >= 3:
            return parts

        def sub_split(items, pattern):
            out = []
            for p in items:
                out.extend(" ".join(x.split()) for x in re.split(pattern, p))
            return [x for x in out if x]

        slash = sub_split(parts, r"\s+/\s+")
        if len(slash) >= 3:
            return slash

        dash = sub_split(slash, r"\s+-\s+")
        if len(dash) >= 3:
            return dash

        kw = []
        for p in dash:
            kw.extend(x for x in self.kw_split_re.split(p) if x.strip())
        kw = [" ".join(x.split()) for x in kw]
        if len(kw) >= 3:
            return kw

        return dash

    # ---------------------------------------------------------
    # PREFIX HANDLING (EXACT + FUZZY PREFIX STRIPPING)

    def _strip_once(self, text: str, words) -> str:
        t = text.strip()
        low = t.lower()
        for w in words:
            if low.startswith(w):
                rest = t[len(w):]
                if not rest:
                    continue
                c = rest[0]
                if c in ". \t" or c.isupper() or (c.isdigit() and w != "t"):
                    return rest.lstrip(" .").strip()
        return t

    def _strip_fuzzy_prefix(self, text: str, p1: tuple, p2: tuple) -> list[str]:
        """
        Cắt tiền tố bị lỗi chính tả (vd: 'ễThành phố Hà Nội', 'Phượờg Nink Kiều',
        'Đọường ặng Dèng', 'éỉnh Nghệ A', 'Thành phbố Hảiự Phòng').
        """
        t = text.strip()
        toks = t.split()
        out = []

        if len(toks) >= 3 and p2:
            head2 = self._compact(toks[0] + toks[1])
            for target in p2:
                if abs(len(head2) - len(target)) <= 2 and self.editdistance.eval(head2, target) <= 2:
                    cand = " ".join(toks[2:]).strip()
                    if cand:
                        out.append(cand)
                    break

        if len(toks) >= 2 and p1:
            head1 = self._compact(toks[0])
            for target in p1:
                md = 1 if len(target) <= 4 else 2
                if abs(len(head1) - len(target)) <= md and self.editdistance.eval(head1, target) <= md:
                    cand = " ".join(toks[1:]).strip()
                    if cand:
                        out.append(cand)
                    break

        if len(toks) >= 1:
            m = re.match(r"^(?:đường|phường|tỉnh|thành\s*phố|phố|xã)([a-zà-ỹ]{1,2}[A-ZÀ-Ỹ].*)$", t, re.I)
            if m:
                out.append(m.group(1).strip())

        return out

    def _strip_chain(self, text: str, words, loop: bool, p1=(), p2=()) -> list[str]:
        chain = [text.strip()]
        for _ in range(3 if loop else 1):
            nxt = self._strip_once(chain[-1], words)
            if nxt == chain[-1] or not nxt:
                break
            chain.append(nxt)

        for extra in self._strip_fuzzy_prefix(chain[-1], p1, p2):
            if extra not in chain:
                chain.append(extra)

        return chain

    # ---------------------------------------------------------
    # FALLBACK

    def _empty_or_raw(self, raw: str) -> str:
        if self.RAW_FALLBACK:
            return raw.strip(" .,;")
        return ""

    # ---------------------------------------------------------
    # PROVINCE MATCHING

    def _province(self, part: str) -> str:
        chain = self._strip_chain(
            part,
            self.prov_prefix,
            loop=True,
            p1=self.prov_fuzzy_1,
            p2=self.prov_fuzzy_2
        )
        for t in chain:
            hit = self._exact(t, self.prov_tab)
            if hit:
                return hit
        for t in reversed(chain):
            hit = self._fuzzy(t, self.prov_tab, self._md)
            if hit:
                return hit
        return self._empty_or_raw(chain[-1])

    # ---------------------------------------------------------
    # WARD MATCHING

    def _ward(self, part: str) -> str:
        chain = self._strip_chain(
            part,
            self.ward_prefix,
            loop=False,
            p1=self.ward_fuzzy_1,
            p2=self.ward_fuzzy_2
        )
        for t in chain:
            hit = self._exact(t, self.ward_tab)
            if hit:
                return hit
        for t in reversed(chain):
            hit = self._fuzzy(t, self.ward_tab, self._md)
            if hit:
                return hit
        return self._empty_or_raw(chain[-1])

    # ---------------------------------------------------------
    # STREET VARIANTS & MATCHING

    def _street_variants(self, part: str):
        p = unicodedata.normalize("NFC", part).strip()
        vs = [p]

        def add(x):
            x = x.strip()
            if x and x not in vs:
                vs.append(x)

        cur = self.km_re.sub("", p, count=1)
        cur = self.kiot_re.sub("", cur, count=1)
        add(cur)

        cur = vs[-1]
        for _ in range(4):
            if not self.alley_re.match(cur):
                break
            cur = self.alley_re.sub("", cur, count=1)
            add(cur)

        add(self.house_re.sub("", vs[-1], count=1))
        add(self.stray_re.sub("", vs[-1], count=1))
        add(self.tail_re.sub("", vs[-1], count=1))

        core_idx = len(vs) - 1
        add(self.prefix_re.sub("", vs[-1], count=1))

        for extra in self._strip_fuzzy_prefix(vs[core_idx], self.street_fuzzy_1, self.street_fuzzy_2):
            add(extra)

        return vs, core_idx

    def _exact_street(self, vs) -> str:
        for v in vs:
            hit = self._exact(v, self.street_tab)
            if hit:
                return hit
        return ""

    def _dict_street(self, vs) -> str:
        hit = self._exact_street(vs)
        if hit:
            return hit
        for v in reversed(vs):
            hit = self._fuzzy(v, self.street_tab, self._md)
            if hit:
                return hit
        return ""

    def _street(self, before: list[str]) -> str:
        cands = []
        for p in reversed(before):
            if self.note_re.match(p):
                continue
            cands.append(self._street_variants(p))

        if not cands:
            return ""

        for vs, ci in cands:
            if self.strong_re.match(vs[ci]):
                hit = self._dict_street(vs)
                if hit:
                    return hit

        for vs, ci in cands:
            hit = self._exact_street(vs)
            if hit:
                return hit

        for vs, ci in cands:
            if not self.hamlet_re.match(vs[ci]):
                hit = self._dict_street(vs)
                if hit:
                    return hit

        if self.RAW_FALLBACK:
            for vs, ci in cands:
                if self.strong_re.match(vs[ci]):
                    return self._empty_or_raw(vs[-1])
            for vs, ci in cands:
                if not self.hamlet_re.match(vs[ci]):
                    return self._empty_or_raw(vs[-1])
            return self._empty_or_raw(cands[0][0][-1])

        return ""

    # ---------------------------------------------------------
    # MAIN PROCESS

    def process(self, s: str) -> dict[str, str]:
        result = {
            "province": "",
            "ward": "",
            "street": ""
        }
        if s is None or not str(s).strip():
            return result

        text = unicodedata.normalize("NFC", str(s))
        text = re.sub(r"\([^)]*\)", " ", text)
        parts = self._split(text)
        if not parts:
            return result

        result["province"] = self._province(parts[-1])

        w_idx = len(parts) - 2
        while w_idx >= 0 and self.district_re.match(parts[w_idx]):
            w_idx -= 1

        if w_idx >= 0:
            result["ward"] = self._ward(parts[w_idx])
            result["street"] = self._street(parts[:w_idx])

        return result

In [75]:
# @title
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt

Downloading...
From: https://drive.google.com/uc?id=18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW
To: /content/test.json
100% 42.7k/42.7k [00:00<00:00, 19.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU
To: /content/list_province.txt
100% 392/392 [00:00<00:00, 1.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h
To: /content/list_ward.txt
100% 1.65k/1.65k [00:00<00:00, 4.90MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa
To: /content/list_street.txt
100% 6.65k/6.65k [00:00<00:00, 19.5MB/s]


In [76]:
def _judge_normalize(s: str, context=None):
    return s

EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json') as f:
    data = json.load(f)

with open("list_province.txt") as f:
    list_province = f.read().strip().splitlines()

with open("list_ward.txt") as f:
    list_ward = f.read().strip().splitlines()

with open("list_street.txt") as f:
    list_street = f.read().strip().splitlines()

summary_only = True
df = []
solution = Solution(
    list_province=list_province,
    list_ward=list_ward,
    list_street=list_street,
)
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["address"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = _judge_normalize(answer["province"])
        answer["ward_normalized"] = _judge_normalize(answer["ward"])
        answer["street_normalized"] = _judge_normalize(answer["street"])

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = _judge_normalize(result["province"])
        result["ward_normalized"] = _judge_normalize(result["ward"])
        result["street_normalized"] = _judge_normalize(result["street"])

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        street_correct = int(answer["street_normalized"] == result["street_normalized"])
        ok = province_correct + street_correct + ward_correct

        df.append([
            test_idx,
            address,

            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,

            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,

            answer["street"],
            result["street"],
            answer["street_normalized"],
            result["street_normalized"],
            street_correct,

            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,

            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,

            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,

            answer["street"],
            "EXCEPTION",
            answer["street_normalized"],
            "EXCEPTION",
            0,

            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',

    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',

    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',

    'street',
    'street_student',
    'street_normalized',
    'street_student_normalized',
    'street_correct',

    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()

------------------------------
TEAM_NAME = 'G_02'
EXCEL_FILE = 'G_02.xlsx'
   correct  total  score / 10  max_time_sec  avg_time_sec
0      600    600        10.0        0.0016        0.0001
